# Tahap 1: Personal Resource Monitor
**Tujuan:** Memantau CPU dan Memory usage laptop sendiri selama beberapa menit,
lalu visualisasikan hasilnya sebagai grafik.

Ini adalah versi mini dari apa yang akan kita lakukan di dataset Alibaba nanti —
konsepnya sama: kumpulkan data resource usage → analisis → visualisasi.

**Yang akan dijawab di notebook ini:**
- Berapa rata-rata CPU dan Memory usage laptop saya?
- Apakah ada lonjakan (spike) di waktu tertentu?
- Berapa gap antara memory yang tersedia vs yang benar-benar dipakai?

## Cell 1: Import Library
Jalankan cell ini pertama. Kalau tidak ada error, berarti semua library sudah siap.

In [ ]:
import psutil
import time
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from datetime import datetime

# Setting style grafik supaya lebih rapi
sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (12, 5)

print('Semua library berhasil diimport!')
print(f'psutil version: {psutil.__version__}')
print(f'pandas version: {pd.__version__}')

## Cell 2: Kumpulkan Data
Cell ini akan merekam CPU dan Memory usage laptop kamu setiap 2 detik, selama 2 menit.

**Selama cell ini berjalan (2 menit):**
- Coba buka beberapa aplikasi, buka browser, putar video — supaya ada variasi data
- Atau biarkan idle dulu sebentar, lalu buka banyak aplikasi di tengah-tengah

Kamu akan lihat output real-time setiap 10 detik.

In [ ]:
# Konfigurasi pengumpulan data
DURASI_DETIK = 120   # Total durasi monitoring (2 menit)
INTERVAL_DETIK = 2   # Ambil data setiap 2 detik

# Tempat menyimpan data
records = []

print(f'Mulai monitoring selama {DURASI_DETIK} detik...')
print(f'Data diambil setiap {INTERVAL_DETIK} detik')
print('-' * 50)

start_time = time.time()
sample_count = 0

while time.time() - start_time < DURASI_DETIK:
    # Ambil data saat ini
    timestamp = datetime.now()
    cpu_pct = psutil.cpu_percent(interval=1)  # % CPU yang dipakai
    mem = psutil.virtual_memory()
    mem_total_gb = mem.total / (1024**3)      # Total RAM dalam GB
    mem_used_gb  = mem.used  / (1024**3)      # RAM yang dipakai dalam GB
    mem_pct      = mem.percent                # % Memory yang dipakai

    # Simpan ke list
    records.append({
        'timestamp'   : timestamp,
        'cpu_pct'     : cpu_pct,
        'mem_used_gb' : mem_used_gb,
        'mem_total_gb': mem_total_gb,
        'mem_pct'     : mem_pct,
        'mem_free_pct': 100 - mem_pct   # % Memory yang TIDAK dipakai (idle)
    })

    sample_count += 1

    # Print update setiap 10 sample
    if sample_count % 5 == 0:
        elapsed = int(time.time() - start_time)
        print(f'[{elapsed:3d}s] CPU: {cpu_pct:5.1f}% | '
              f'Memory: {mem_used_gb:.1f}/{mem_total_gb:.1f} GB ({mem_pct:.1f}%)')

    time.sleep(INTERVAL_DETIK)

# Konversi ke DataFrame pandas
df = pd.DataFrame(records)

print('-' * 50)
print(f'Selesai! Total data terkumpul: {len(df)} baris')

## Cell 3: Statistik Dasar
Ini langkah pertama yang selalu dilakukan sebelum buat grafik apapun:
lihat ringkasan statistik datanya dulu.

In [ ]:
print('=== STATISTIK DASAR ===')
print()

# Statistik CPU
print('--- CPU Usage (%) ---')
print(f'Rata-rata : {df["cpu_pct"].mean():.1f}%')
print(f'Minimum   : {df["cpu_pct"].min():.1f}%')
print(f'Maximum   : {df["cpu_pct"].max():.1f}%')
print(f'Median    : {df["cpu_pct"].median():.1f}%')
print()

# Statistik Memory
print('--- Memory Usage ---')
print(f'Total RAM          : {df["mem_total_gb"].mean():.1f} GB')
print(f'Rata-rata dipakai  : {df["mem_used_gb"].mean():.1f} GB ({df["mem_pct"].mean():.1f}%)')
print(f'Rata-rata idle     : {df["mem_total_gb"].mean() - df["mem_used_gb"].mean():.1f} GB ({df["mem_free_pct"].mean():.1f}%)')
print(f'Pemakaian tertinggi: {df["mem_used_gb"].max():.1f} GB ({df["mem_pct"].max():.1f}%)')
print()

# Gap analysis - ini yang paling relevan ke riset kita!
avg_mem_idle = df['mem_free_pct'].mean()
print('--- Gap Analysis (Preview RQ1) ---')
print(f'Rata-rata memory yang TIDAK dipakai: {avg_mem_idle:.1f}%')
print(f'Artinya dari {df["mem_total_gb"].mean():.1f} GB RAM,')
print(f'sekitar {df["mem_total_gb"].mean() * avg_mem_idle/100:.1f} GB rata-rata idle/tidak terpakai')
print()
print('(Ini versi mini dari analisis over-provisioning yang akan')
print('kita lakukan di dataset Alibaba 4000 mesin nanti)')

## Cell 4: Visualisasi — Time Series
Grafik pertama: CPU dan Memory usage dari waktu ke waktu.
Ini versi mini dari Figure 2 dan Figure 4 di paper Guo et al. (2019).

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# --- Plot CPU ---
ax1.plot(df['timestamp'], df['cpu_pct'],
         color='steelblue', linewidth=1.5, label='CPU Usage')
ax1.axhline(y=df['cpu_pct'].mean(), color='red',
            linestyle='--', linewidth=1, label=f'Rata-rata ({df["cpu_pct"].mean():.1f}%)')
ax1.set_ylabel('CPU Usage (%)')
ax1.set_ylim(0, 100)
ax1.legend(loc='upper right')
ax1.set_title('CPU dan Memory Usage Over Time (Laptop Lokal)', fontsize=13)

# --- Plot Memory ---
ax2.plot(df['timestamp'], df['mem_pct'],
         color='darkorange', linewidth=1.5, label='Memory Usage')
ax2.axhline(y=df['mem_pct'].mean(), color='red',
            linestyle='--', linewidth=1, label=f'Rata-rata ({df["mem_pct"].mean():.1f}%)')
ax2.fill_between(df['timestamp'], df['mem_pct'], alpha=0.2, color='darkorange')
ax2.set_ylabel('Memory Usage (%)')
ax2.set_xlabel('Waktu')
ax2.set_ylim(0, 100)
ax2.legend(loc='upper right')

# Format sumbu waktu
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
plt.xticks(rotation=45)

plt.tight_layout()
plt.savefig('figure1_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafik disimpan sebagai figure1_timeseries.png')

## Cell 5: Visualisasi — Distribusi (Histogram)
Grafik kedua: seberapa sering CPU/Memory berada di range tertentu.
Ini versi mini dari Figure 3 di paper Guo et al. (2019).

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# --- Histogram CPU ---
ax1.hist(df['cpu_pct'], bins=20, color='steelblue',
         edgecolor='white', alpha=0.8)
ax1.axvline(x=df['cpu_pct'].mean(), color='red',
            linestyle='--', label=f'Mean: {df["cpu_pct"].mean():.1f}%')
ax1.set_xlabel('CPU Usage (%)')
ax1.set_ylabel('Frekuensi')
ax1.set_title('Distribusi CPU Usage')
ax1.legend()

# --- Histogram Memory ---
ax2.hist(df['mem_pct'], bins=20, color='darkorange',
         edgecolor='white', alpha=0.8)
ax2.axvline(x=df['mem_pct'].mean(), color='red',
            linestyle='--', label=f'Mean: {df["mem_pct"].mean():.1f}%')
ax2.set_xlabel('Memory Usage (%)')
ax2.set_ylabel('Frekuensi')
ax2.set_title('Distribusi Memory Usage')
ax2.legend()

plt.suptitle('Distribusi Resource Usage - Laptop Lokal', fontsize=13)
plt.tight_layout()
plt.savefig('figure2_distribusi.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafik disimpan sebagai figure2_distribusi.png')

## Cell 6: Visualisasi — Gap Analysis (Preview RQ1 Paper)
Grafik ketiga: visualisasi 'gap' antara resource yang TERSEDIA vs yang DIPAKAI.
Ini adalah konsep inti yang akan menjadi RQ1 di paper kita nanti.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

# Area yang dipakai (bawah)
ax.fill_between(df['timestamp'], df['mem_pct'],
                alpha=0.7, color='darkorange', label='Memory Dipakai')

# Area yang idle/tidak dipakai (atas) — ini yang disebut "gap" atau "waste"
ax.fill_between(df['timestamp'], df['mem_pct'], 100,
                alpha=0.3, color='gray', label='Memory Idle (tidak terpakai)')

# Garis rata-rata
ax.axhline(y=df['mem_pct'].mean(), color='red',
           linestyle='--', linewidth=1.5,
           label=f'Rata-rata dipakai: {df["mem_pct"].mean():.1f}%')

ax.set_ylabel('Memory (%)')
ax.set_xlabel('Waktu')
ax.set_ylim(0, 100)
ax.set_title('Gap Analysis: Memory Dipakai vs Idle', fontsize=13)
ax.legend(loc='lower right')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
plt.xticks(rotation=45)

# Tambahkan anotasi gap rata-rata
avg_idle = df['mem_free_pct'].mean()
ax.annotate(f'Rata-rata idle:\n{avg_idle:.1f}%',
            xy=(df['timestamp'].iloc[len(df)//2], 100 - avg_idle/2),
            fontsize=11, color='gray',
            ha='center', va='center')

plt.tight_layout()
plt.savefig('figure3_gap_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafik disimpan sebagai figure3_gap_analysis.png')

## Cell 7: Simpan Data ke CSV
Simpan data mentah ke file CSV — ini kebiasaan baik dalam riset:
selalu simpan data mentah terpisah dari hasil analisis.

In [ ]:
# Simpan ke CSV
df.to_csv('resource_monitor_data.csv', index=False)
print('Data disimpan ke resource_monitor_data.csv')
print()
print('Preview 5 baris pertama:')
print(df.head())
print()
print('Shape data:', df.shape)
print('Kolom:', list(df.columns))

## Cell 8: Ringkasan & Koneksi ke Riset
Ini bagian penting: selalu akhiri dengan interpretasi hasil
dan koneksinya ke pertanyaan riset yang lebih besar.

In [ ]:
print('=' * 55)
print('RINGKASAN HASIL - TAHAP 1')
print('=' * 55)
print()
print(f'Durasi monitoring : {DURASI_DETIK} detik')
print(f'Total data points : {len(df)} samples')
print()
print('CPU:')
print(f'  Rata-rata usage : {df["cpu_pct"].mean():.1f}%')
print(f'  Artinya idle    : {100 - df["cpu_pct"].mean():.1f}% dari kapasitas CPU')
print()
print('Memory:')
print(f'  Total           : {df["mem_total_gb"].mean():.1f} GB')
print(f'  Rata-rata dipakai: {df["mem_used_gb"].mean():.1f} GB ({df["mem_pct"].mean():.1f}%)')
print(f'  Rata-rata idle  : {df["mem_total_gb"].mean() - df["mem_used_gb"].mean():.1f} GB ({df["mem_free_pct"].mean():.1f}%)')
print()
print('-' * 55)
print('KONEKSI KE RISET (Alibaba Cluster nanti):')
print('-' * 55)
print()
print('Skala yang berbeda:')
print(f'  Laptop ini    : 1 mesin, {df["mem_total_gb"].mean():.0f} GB RAM')
print(f'  Alibaba trace : 4.000 mesin, masing-masing 96 CPU cores')
print()
print('Tapi konsepnya SAMA:')
print('  - Kita ukur berapa yang dialokasikan vs yang dipakai')
print('  - Kita hitung gap/idle-nya')
print('  - Kita visualisasikan polanya dari waktu ke waktu')
print()
print('Di Alibaba trace nanti, kita tambahkan:')
print('  - Estimasi energi yang terbuang dari gap itu (RQ3)')
print('  - Analisis pola waktu di 4.000 mesin selama 8 hari (RQ2)')
print('=' * 55)